In [266]:
import sys
# sys.path.append("/Users/bubble/Desktop/Project/Infrasound Sensor/Layout/Code/acousticsensor")
import math
import gdsfactory as gf
cell_temp = gf.Component()
# the whole chip
block = gf.Component()

In [267]:
# note
"""
layer 9: structure to keep
layer 11: gold deposition area
layer 12: frontside etching area
"""

'\nlayer 9: structure to keep\nlayer 11: gold deposition area\nlayer 12: frontside etching area\n'

In [268]:
# frontside etching area
frame_size = [500, 1000, 2000]
frame_numbe = [6, 4, 3]
gap = [2000, 3000, 4000]
origin_x = [-5000, -4500, -4000]
origin_y = [-4000, -1000, 2500]
for i in range(len(frame_size)):
    for j in range(frame_numbe[i]):
        frontside = gf.components.rectangle(size=(frame_size[i], frame_size[i]), layer=(12, 0))
        backside_size = frame_size[i] + 743.44
        backside = gf.components.rectangle(size=(backside_size, backside_size), layer=(3, 0))
        (block << frontside).move((-frame_size[i]/2+gap[i]*j+origin_x[i], -frame_size[i]/2+origin_y[i]))
        (block << backside).move((-backside_size/2+gap[i]*j+origin_x[i], -backside_size/2+origin_y[i]))
        # length mark
        T = gf.components.text(f"L={frame_size[i]}", size=50, layer=(1, 0))
        (block << T).move((gap[i]*j+origin_x[i]-100, -frame_size[i]/2+origin_y[i]-200))

# side frame etching area
# 7300um x 7300um gap between frame: 600
etch_frame = gf.components.rectangle(size=(13025, 375), layer=(3, 0))
for i in range(4):
    angle = 90 * i
    (block << etch_frame).move((-13025/2, -14225/2)).rotate(angle=angle, center=(0, 0))
# block.show()

In [ ]:
# order
def add_order_text(order_number):
    order = gf.Component()
    for j in range(4):
        T = gf.components.text(f"FT{order_number}", size=20, layer=(1, 0))
        order_ref = order << T
        if j == 0:
            order_ref.move((-5000, -5000))
        elif j == 1:
            order_ref.move((-5000, 4500))
        elif j == 2:
            order_ref.move((5000, -5000))
        else:
            order_ref.move((5000, 4500))
    return order
    # order.show()
        

In [270]:
# repeat
fblock = gf.Component()
block_temp = gf.Component()
block_temp << block
for i in range(1):
    if i == 0:
        block_ref = fblock << block_temp
    elif i == 1:
        block_ref = fblock << block_temp
        block_ref.move((10000, 0))
    elif i == 2:
        block_ref = fblock << block_temp
        block_ref.move((10000, 10000))
    else:
        block_ref = fblock << block_temp
        block_ref.move((0, 10000))

In [ ]:
# boolean operation
outside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(12, 0), layer2=(9, 0), layer=(1, 0))
cell_temp << outside

#  gold
gold = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(11, 0), layer2=(30, 0), layer=(2, 0))
cell_temp << gold
# backside etching
backside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(3, 0), layer2=(30, 0), layer=(3, 0))
cell_temp << backside
# add marker/order
marker = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(1, 0), layer2=(30, 0), layer=(1, 0))
cell_temp << marker

def cell_frame_test(order_number):
    cell_frame_test = gf.Component()
    cell_temp_temp = gf.Component()
    order = add_order_text(order_number)
    cell_temp_temp << order
    cell_temp_temp << cell_temp
    cell_frame_test << cell_temp_temp
    return cell_frame_test
    # cell_ref.move((2500, -7500))
    # cell_frame_test.show()
    # cell_frame_test.write_gds("mesh.gds")
    # cell_frame_test.plot()

if __name__ == "__main__":
    cell = cell_frame_test(order_number=1)
    cell.show()